# Adult Census Income: mixed-type pipeline in OMLE

Demonstrate a raw DataFrame containing **integer, floating-point, and string columns**, including missing values. Fit a `ColumnTransformer` and LogisticRegression together, export the complete pipeline, inspect its interface and operators, validate, view, and verify predictions.

## Dataset and input contract

Load pinned [OpenML dataset 1590](https://www.openml.org/d/1590) through `sklearn.datasets.fetch_openml`. The first run needs internet; subsequent runs reuse `.cache/openml/`, excluded from Git. The returned metadata below records the dataset version, source checksum, attribution, and license. No dataset is committed or uploaded.

Use a deliberate subset of available predictors to keep the graph readable. Integer columns remain integer; continuous numeric columns are explicitly float64, including numeric measurements stored as integers by the source. Categorical columns are Python strings (`object` dtype), with `np.nan` for missing entries. These type and missing-marker normalizations define the input contract for both native and OMLE predictions; learned imputation, scaling, and encoding remain inside the exported graph.
This is an educational portability example, not a deployment recommendation.

**Setup:** install `requirements.txt` in the notebook kernel environment.

In [ ]:
import hashlib
import json
import platform
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.datasets import fetch_openml
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, classification_report, log_loss, roc_auc_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import omle
import omle_runtime as omr
from omle_convert import export_omle

SEED = 42
ATOL = 1e-4
RTOL = 1e-4
DATA_ID = 1590
integer_columns = ['age', 'education-num']
float_columns = ['capital-gain', 'capital-loss', 'hours-per-week']
categorical_columns = ['workclass', 'education', 'marital-status', 'occupation', 'native-country']
numeric_columns = integer_columns + float_columns
feature_columns = numeric_columns + categorical_columns
packages = ["numpy", "pandas", "scikit-learn", "omle", "omle-convert", "omle-runtime", "omle-viewer"]
environment = {"python": platform.python_version(), "packages": {p: version(p) for p in packages}}
cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents]
                  if (p / "requirements.txt").is_file() and (p / "notebooks").is_dir()), None)
if repo_root is None:
    raise RuntimeError("Start Jupyter from omle-models or a notebook subdirectory.")
display(environment)

## Load and inspect mixed types

Download by immutable dataset ID. Preserve the returned source metadata and compute a SHA-256 fingerprint of the normalized selected data and target. Display the input dtypes, example values, and missing-value counts before fitting.

In [ ]:
cache_dir = repo_root / ".cache" / "openml"
# Read source fields as strings; apply the explicit input contract below.
dataset = fetch_openml(data_id=DATA_ID, as_frame=True, parser="pandas", data_home=str(cache_dir),
                       read_csv_kwargs={"dtype": str})
source_metadata = {key: dataset.details.get(key) for key in
                   ["id", "name", "version", "licence", "creator", "citation", "url", "md5_checksum"]}
display(source_metadata)
X = dataset.data.loc[:, feature_columns].copy()
for column in integer_columns:
    values = pd.to_numeric(X[column], errors="raise")
    assert values.notna().all() and (values == np.floor(values)).all()
    X[column] = values.astype(np.int64)
for column in float_columns:
    X[column] = pd.to_numeric(X[column], errors="raise").astype(np.float64)
for column in categorical_columns:
    values = X[column].astype(object)
    values = values.map(lambda value: str(value).strip() if pd.notna(value) else np.nan)
    X[column] = values.replace("?", np.nan).astype(object)
target_values = dataset.target.astype(str).str.strip().str.rstrip(".")
assert set(target_values.unique()) == {"<=50K", ">50K"}
y = target_values.map({"<=50K": 0, ">50K": 1}).astype(np.int64)
class_names = ["income <=50K", "income >50K"]
assert not y.isna().any()
input_summary = pd.DataFrame({"dtype": X.dtypes.astype(str), "missing_count": X.isna().sum()})
display(input_summary)
display(X.head())
normalized_data = X.assign(__target__=y)
dataset_sha256 = hashlib.sha256(pd.util.hash_pandas_object(normalized_data, index=True).to_numpy().tobytes()).hexdigest()
print(f"Rows: {len(X)}; selected features: {len(X.columns)}; normalized data SHA-256: {dataset_sha256}")

## Split and fit preprocessing

Use a fixed 80/20 split stratified by class. Fit all learned preprocessing on training rows only:

- Numeric features: median imputation followed by standard scaling.
- String features: dense one-hot encoding, with unknown categories producing all-zero columns for that feature.

Missing categorical values are encoded as their own category when observed in training; otherwise they follow the unknown-category rule. No string imputer is required. This policy also applies to the edge-case checks below.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_columns),
    ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_columns),
], remainder="drop", sparse_threshold=0)
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, random_state=SEED)),
])
pipeline.fit(X_train, y_train)
print(f"Training rows: {len(X_train)}; held-out rows: {len(X_test)}")

## Evaluate the native pipeline

Quality metrics describe the held-out prediction task. They do not establish conversion correctness; that is verified independently below.

In [ ]:
native_labels = pipeline.predict(X_test)
native_probabilities = pipeline.predict_proba(X_test)
np.testing.assert_array_equal(pipeline.classes_, [0, 1])
display(pd.DataFrame({"probability_column": [0, 1], "label": pipeline.classes_, "class_name": class_names}))
metrics = {
    "accuracy": float(accuracy_score(y_test, native_labels)),
    "log_loss": float(log_loss(y_test, native_probabilities, labels=pipeline.classes_)),
    "roc_auc_class_1": float(roc_auc_score(y_test, native_probabilities[:, 1])),
}
display(pd.Series(metrics, name="held_out_metrics"))
print(classification_report(y_test, native_labels, target_names=class_names, zero_division=0))

## Export the fitted pipeline

Pass the raw mixed-type training DataFrame to the converter so it can infer per-column input specifications and embed verification samples. The graph contains both preprocessing branches, their concatenation, and the fitted estimator.

In [ ]:
output_dir = repo_root / "models" / "classification" / "adult"
output_dir.mkdir(parents=True, exist_ok=True)
model_path = output_dir / "sklearn-mixed-logistic-regression.omle"
export_omle(
    pipeline, str(model_path), X=X_train,
    model_name="adult-sklearn-mixed-logistic-regression", model_version="0.1.0",
    random_state=SEED, n_verify=10, n_sample=3, verify_atol=ATOL, verify_rtol=RTOL,
)
print(f"Exported {model_path.relative_to(repo_root)} ({model_path.stat().st_size:,} bytes)")

## Inspect the exported model

Show the mixed-type input contract and output specifications. The graph accepts raw columns, not a pre-encoded numeric matrix. A dimension of `-1` represents a variable batch size.

In [ ]:
import omle

exported_model = omle.load(str(model_path))

print("Model inputs")
display(pd.DataFrame([
    {
        "name": spec.name,
        "dtype": spec.type.dtype.name if spec.type else None,
        "shape": spec.type.shape if spec.type else None,
    }
    for spec in exported_model.inputs
]))

print("Model outputs")
display(pd.DataFrame([
    {
        "name": spec.name,
        "dtype": spec.type.dtype.name if spec.type else None,
        "shape": spec.type.shape if spec.type else None,
        "role": spec.role.name,
    }
    for spec in exported_model.outputs
]))

### Operators included in the model

List actual graph operators, including nodes nested inside composite graphs when present.

In [ ]:
operator_versions = {item.namespace: item.version for item in exported_model.operator_imports}

def operator_rows(nodes, parent=""):
    for node in nodes:
        path = f"{parent}/{node.name}" if parent else node.name
        yield {"node": path, "namespace": node.domain, "operator": node.op,
               "version": operator_versions.get(node.domain)}
        if node.composite is not None:
            yield from operator_rows(node.composite.nodes, path)

operators = pd.DataFrame(operator_rows(exported_model.nodes))
display(operators)

### Validate the model

Structural validation checks the exported graph and operator declarations; errors stop execution. Numerical parity is checked below.

In [ ]:
validation = omle.validate(exported_model)
print(validation)
if not validation.is_valid:
    raise ValueError(f"Exported OMLE model is invalid:\n{validation}")

### Explore the interactive graph

Inspect the imputer, scaler, encoder, concatenation, and model nodes. Run the cell in JupyterLab or your IDE notebook editor; the viewer does not execute in GitHub’s static preview. No model upload is needed.

In [ ]:
from omle_viewer import show

viewer = show(exported_model, height=650)

## Verify raw DataFrame predictions

Send the same raw mixed-type DataFrame directly to both implementations, including its missing values. Require matching shapes, finite values, exact classification labels, and numeric agreement at the stated tolerances. Class probability order is checked against the native estimator above.

In [ ]:
runtime_model = omr.load(str(model_path))

def verify_predictions(frame):
    native = pipeline.predict(frame)
    actual = runtime_model.predict(frame)
    assert actual.shape == native.shape
    assert np.isfinite(actual).all()
    np.testing.assert_array_equal(actual, native)
    expected_proba = pipeline.predict_proba(frame)
    actual_proba = runtime_model.predict_proba(frame)
    assert actual_proba.shape == expected_proba.shape
    assert np.isfinite(actual_proba).all()
    assert ((actual_proba >= 0) & (actual_proba <= 1)).all()
    np.testing.assert_allclose(actual_proba, expected_proba, atol=ATOL, rtol=RTOL)
    np.testing.assert_allclose(actual_proba.sum(axis=1), 1.0, atol=ATOL, rtol=RTOL)
    return {"rows": len(frame), "label_agreement": float(np.mean(actual == native)),
            "max_absolute_probability_error": float(np.max(np.abs(actual_proba - expected_proba)))}
parity = verify_predictions(X_test)
parity.update({"atol": ATOL, "rtol": RTOL})
display(pd.Series(parity, name="held_out_conversion_fidelity"))
print("PASS: raw mixed-type test predictions match.")

## Check missing values and unseen categories

Construct separate probes from held-out rows without refitting. Inject a missing float, a missing string, and an unseen string category, then compare both inference paths. These are conversion checks, not additional quality measurements. Integer columns remain integers; missing numeric values are inserted into the floating-point branch.

In [ ]:
probes = pd.concat([X_test.iloc[[0]].copy() for _ in range(4)], ignore_index=True)
probe_names = ["baseline", "missing_numeric", "missing_categorical", "unseen_category"]
probes.loc[1, float_columns[0]] = np.nan
probes.loc[2, categorical_columns[0]] = np.nan
unknown_value = "__OMLE_UNSEEN_CATEGORY__"
assert unknown_value not in set(X_train[categorical_columns[0]].dropna())
probes.loc[3, categorical_columns[0]] = unknown_value
display(probes.assign(probe=probe_names))
edge_parity = verify_predictions(probes)
display(pd.Series(edge_parity, name="edge_case_conversion_fidelity"))
print("PASS: missing values and unseen categories match the native pipeline.")

## Save the run record

Record source identity/checksum, normalized-data fingerprint, column dtypes and selections, exact dependencies, split membership, fitted configuration, and both parity checks. Files remain local and ignored until explicitly selected for publication.

In [ ]:
report = {
    "environment": environment, "dataset": source_metadata,
    "normalized_dataset_sha256": dataset_sha256,
    "seed": SEED, "test_size": 0.2,
    "feature_names": list(X.columns), "input_dtypes": X.dtypes.astype(str).to_dict(),
    "integer_columns": integer_columns, "float_columns": float_columns,
    "categorical_columns": categorical_columns,
    "categorical_missing_policy": "NaN category if seen in training; otherwise unknown all-zero encoding",
    "train_indices": X_train.index.tolist(), "test_indices": X_test.index.tolist(),
    "model_parameters": {str(k): str(v) for k, v in pipeline.get_params(deep=True).items()},
    "metrics": metrics, "parity": parity, "edge_parity": edge_parity,
}
report["class_names"] = class_names
report["classes"] = pipeline.classes_.tolist()
report_path = output_dir / "sklearn-mixed-logistic-regression-environment.json"
report_path.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print(f"Saved {report_path.relative_to(repo_root)}")